# Module 4: Professional GA Implementation with DEAP

## Learning Objectives

By completing this notebook, you will:
1. Master the DEAP library architecture and components
2. Implement complete feature selection GAs with DEAP
3. Use DEAP's built-in algorithms (eaSimple, eaMuPlusLambda)
4. Implement custom evolution strategies
5. Leverage DEAP's tools: HallOfFame, Statistics, Logbook
6. Integrate custom operators and fitness functions
7. Export results and create production-ready pipelines

## Prerequisites

- Module 1 completed (GA fundamentals)
- Module 2 completed (fitness functions)
- DEAP library installed (`pip install deap`)

## Estimated Time: 90 minutes

---

## 1. DEAP Architecture Overview

### Key Concept: DEAP uses a registration-based system.

**Core Components:**
1. **Creator**: Define custom types (Fitness, Individual)
2. **Toolbox**: Register functions (operators, evaluation)
3. **Algorithms**: High-level evolution strategies
4. **Tools**: Statistics, HallOfFame, Logbook

**Workflow:**
```python
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, ...)
toolbox.register("evaluate", my_fitness_function)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)
```

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

# DEAP imports
from deap import base, creator, tools, algorithms
import random

# Set random seeds
np.random.seed(42)
random.seed(42)

# Plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("All libraries imported successfully!")
print(f"DEAP version: {getattr(tools, '__version__', 'version not available')}")

### 1.1 Load and Prepare Dataset

In [ ]:
# Purpose: Prepare dataset for DEAP-based feature selection
# Key Concept: Global data accessible to fitness function

# Load data
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X.columns,
    index=X_test.index
)

print(f"Dataset: {X_train_scaled.shape[0]} train, {X_test_scaled.shape[0]} test")
print(f"Features: {X_train_scaled.shape[1]}")
print(f"Classes: {np.unique(y, return_counts=True)}")

## 2. Basic DEAP Setup for Feature Selection

### Key Concept: Define custom types once, use throughout the session.

### 2.1 Create Custom Types

In [ ]:
# Purpose: Create DEAP custom types for feature selection
# Key Concept: FitnessMax for maximization, Individual as binary list

# WARNING: Only run once per session!
# DEAP caches types globally

# Clean up if types already exist (for notebook re-running)
if hasattr(creator, "FitnessMax"):
    del creator.FitnessMax
if hasattr(creator, "Individual"):
    del creator.Individual

# Create fitness class (maximize single objective)
creator.create("FitnessMax", base.Fitness, weights=(1.0,))

# Create individual class (list with fitness attribute)
creator.create("Individual", list, fitness=creator.FitnessMax)

print("Custom types created:")
print(f"  FitnessMax: {creator.FitnessMax}")
print(f"  Individual: {creator.Individual}")
print(f"  Fitness weights: {creator.FitnessMax.weights} (positive = maximize)")

### 2.2 Define Fitness Function

In [ ]:
# Purpose: Define fitness function for feature selection
# Key Concept: Must return tuple (DEAP requirement)

def evaluate_features(individual):
    """
    Evaluate fitness of a binary feature selection individual.
    
    Parameters
    ----------
    individual : list of int
        Binary chromosome (1 = selected, 0 = not selected)
    
    Returns
    -------
    fitness : tuple of float
        Single-element tuple with fitness value (DEAP requirement)
    """
    # Convert to numpy array
    individual_array = np.array(individual)
    
    # Check if at least one feature selected
    if np.sum(individual_array) == 0:
        return (-999.0,)  # Very bad fitness
    
    # Select features
    feature_mask = individual_array.astype(bool)
    X_selected = X_train_scaled.iloc[:, feature_mask]
    
    # Evaluate with cross-validation
    model = LogisticRegression(max_iter=1000, random_state=42)
    scores = cross_val_score(
        model, X_selected, y_train,
        cv=5,
        scoring='accuracy'
    )
    
    accuracy = scores.mean()
    
    # Apply parsimony pressure
    parsimony_penalty = 0.01 * (np.sum(individual_array) / len(individual_array))
    
    # Combined fitness
    fitness = accuracy - parsimony_penalty
    
    # Must return tuple!
    return (fitness,)

# Test fitness function
test_individual = [1, 1, 1, 0, 0, 0, 0, 0, 0, 0] + [0] * (X_train_scaled.shape[1] - 10)
test_fitness = evaluate_features(test_individual)
print(f"Test fitness: {test_fitness[0]:.4f}")
print(f"Selected {sum(test_individual)} features")

### 2.3 Register Toolbox Components

In [ ]:
# Purpose: Register all GA components in toolbox
# Key Concept: Toolbox centralizes all operators

# Create toolbox
toolbox = base.Toolbox()

# Register attribute generator (random binary)
toolbox.register("attr_bool", random.randint, 0, 1)

# Register individual generator
N_FEATURES = X_train_scaled.shape[1]
toolbox.register(
    "individual",
    tools.initRepeat,
    creator.Individual,
    toolbox.attr_bool,
    n=N_FEATURES
)

# Register population generator
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# Register genetic operators
toolbox.register("evaluate", evaluate_features)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)

print("Toolbox registered:")
print(f"  Individual length: {N_FEATURES}")
print(f"  Evaluation: {toolbox.evaluate}")
print(f"  Crossover: {toolbox.mate}")
print(f"  Mutation: {toolbox.mutate} (indpb=0.05)")
print(f"  Selection: {toolbox.select} (tournsize=3)")

## 3. Running the GA with DEAP

### Key Concept: DEAP provides ready-to-use evolution algorithms.

### 3.1 Setup Statistics and Hall of Fame

Track evolution progress and preserve best individuals.

In [ ]:
# Purpose: Configure statistics tracking and hall of fame
# Key Concept: Statistics object defines what to track

# Statistics object
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)

# Hall of Fame: keep best N individuals
hall_of_fame = tools.HallOfFame(maxsize=10)

print("Statistics configured:")
print(f"  Tracking: avg, std, min, max")
print(f"  Hall of Fame size: 10")

### 3.2 Run Simple GA (eaSimple)

Use DEAP's built-in generational GA.

In [ ]:
# Purpose: Run standard generational GA
# Key Concept: eaSimple implements classic GA with elitism

# GA parameters
POPULATION_SIZE = 50
P_CROSSOVER = 0.7
P_MUTATION = 0.2
MAX_GENERATIONS = 40

# Create initial population
population = toolbox.population(n=POPULATION_SIZE)

print(f"Running GA with eaSimple...")
print(f"  Population: {POPULATION_SIZE}")
print(f"  Crossover prob: {P_CROSSOVER}")
print(f"  Mutation prob: {P_MUTATION}")
print(f"  Generations: {MAX_GENERATIONS}")
print("\nEvolution progress:")

# Run evolution
population, logbook = algorithms.eaSimple(
    population,
    toolbox,
    cxpb=P_CROSSOVER,
    mutpb=P_MUTATION,
    ngen=MAX_GENERATIONS,
    stats=stats,
    halloffame=hall_of_fame,
    verbose=True
)

print("\n" + "="*70)
print("Evolution completed!")

### 3.3 Analyze Results

In [ ]:
# Purpose: Extract and display best solutions
# Key Concept: Hall of Fame stores best individuals

print("\nTop 5 Solutions:")
print("="*70)

for i, ind in enumerate(hall_of_fame[:5], 1):
    n_features = sum(ind)
    fitness = ind.fitness.values[0]
    selected_idx = [j for j, gene in enumerate(ind) if gene == 1]
    selected_names = X_train_scaled.columns[selected_idx].tolist()
    
    print(f"\n{i}. Fitness: {fitness:.4f}")
    print(f"   Features: {n_features}/{N_FEATURES}")
    print(f"   Selected: {selected_names[:5]}{'...' if len(selected_names) > 5 else ''}")

# Best solution
best_individual = hall_of_fame[0]
best_features = [i for i, gene in enumerate(best_individual) if gene == 1]
best_feature_names = X_train_scaled.columns[best_features].tolist()

print("\n" + "="*70)
print("Best Solution Details:")
print(f"  Fitness: {best_individual.fitness.values[0]:.4f}")
print(f"  Features selected: {len(best_features)}")
print(f"  Feature names: {best_feature_names}")

### 3.4 Visualize Evolution

In [ ]:
# Purpose: Visualize evolution using logbook data
# Key Concept: Logbook stores statistics history

# Extract data from logbook
gen = logbook.select("gen")
fit_avg = logbook.select("avg")
fit_max = logbook.select("max")
fit_min = logbook.select("min")
fit_std = logbook.select("std")

# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Fitness evolution
axes[0, 0].plot(gen, fit_max, 'r-', linewidth=2, label='Best')
axes[0, 0].plot(gen, fit_avg, 'b-', linewidth=2, label='Average')
axes[0, 0].plot(gen, fit_min, 'g-', linewidth=2, label='Worst')
axes[0, 0].set_xlabel('Generation', fontsize=12)
axes[0, 0].set_ylabel('Fitness', fontsize=12)
axes[0, 0].set_title('Fitness Evolution', fontsize=14)
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Plot 2: Population diversity
axes[0, 1].plot(gen, fit_std, 'purple', linewidth=2)
axes[0, 1].set_xlabel('Generation', fontsize=12)
axes[0, 1].set_ylabel('Fitness Std Dev', fontsize=12)
axes[0, 1].set_title('Population Diversity', fontsize=14)
axes[0, 1].grid(alpha=0.3)

# Plot 3: Convergence rate
improvement = np.diff(fit_max)
axes[1, 0].plot(gen[1:], improvement, 'orange', linewidth=2)
axes[1, 0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1, 0].set_xlabel('Generation', fontsize=12)
axes[1, 0].set_ylabel('Best Fitness Improvement', fontsize=12)
axes[1, 0].set_title('Convergence Rate', fontsize=14)
axes[1, 0].grid(alpha=0.3)

# Plot 4: Best solution features
axes[1, 1].bar(range(len(best_individual)), best_individual, 
               color='steelblue', edgecolor='black')
axes[1, 1].set_xlabel('Feature Index', fontsize=12)
axes[1, 1].set_ylabel('Selected (1) or Not (0)', fontsize=12)
axes[1, 1].set_title(f'Best Solution ({sum(best_individual)} features)', fontsize=14)
axes[1, 1].set_ylim(-0.1, 1.1)
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Evolution Analysis:")
print(f"  Initial best fitness: {fit_max[0]:.4f}")
print(f"  Final best fitness: {fit_max[-1]:.4f}")
print(f"  Total improvement: {fit_max[-1] - fit_max[0]:.4f}")
print(f"  Convergence generation: ~{np.argmax(np.array(improvement) < 0.001)}")

## 4. Advanced DEAP Features

### Key Concept: DEAP supports advanced evolution strategies beyond eaSimple.

### 4.1 Mu+Lambda Evolution Strategy

Keep best mu individuals, generate lambda offspring, select best mu from combined pool.

In [ ]:
# Purpose: Implement (mu+lambda) evolution strategy
# Key Concept: More elitist than standard GA

# Reset hall of fame and stats
hall_of_fame_mu = tools.HallOfFame(maxsize=10)
stats_mu = tools.Statistics(key=lambda ind: ind.fitness.values)
stats_mu.register("avg", np.mean)
stats_mu.register("max", np.max)

# Create new population
population_mu = toolbox.population(n=50)

# Mu+Lambda parameters
MU = 50  # Number of individuals to keep
LAMBDA = 100  # Number of offspring to generate
NGEN = 40

print(f"Running (mu+lambda) ES...")
print(f"  Mu: {MU}")
print(f"  Lambda: {LAMBDA}")
print(f"  Generations: {NGEN}")
print("\nEvolution progress:")

# Run evolution
population_mu, logbook_mu = algorithms.eaMuPlusLambda(
    population_mu,
    toolbox,
    mu=MU,
    lambda_=LAMBDA,
    cxpb=0.7,
    mutpb=0.2,
    ngen=NGEN,
    stats=stats_mu,
    halloffame=hall_of_fame_mu,
    verbose=True
)

print("\n(mu+lambda) ES completed!")
print(f"Best fitness: {hall_of_fame_mu[0].fitness.values[0]:.4f}")
print(f"Features: {sum(hall_of_fame_mu[0])}/{N_FEATURES}")

### 4.2 Multi-Objective Optimization (NSGA-II)

Find Pareto front for accuracy vs number of features.

In [ ]:
# Purpose: Implement multi-objective optimization with NSGA-II
# Key Concept: Optimize multiple objectives simultaneously

# Clean up previous types
if hasattr(creator, "FitnessMulti"):
    del creator.FitnessMulti
if hasattr(creator, "IndividualMulti"):
    del creator.IndividualMulti

# Create multi-objective fitness (maximize both)
creator.create("FitnessMulti", base.Fitness, weights=(1.0, 1.0))
creator.create("IndividualMulti", list, fitness=creator.FitnessMulti)

# Multi-objective fitness function
def evaluate_multi_objective(individual):
    """
    Evaluate two objectives:
    1. Maximize accuracy
    2. Maximize parsimony (minimize features, so return negative)
    
    Returns
    -------
    objectives : tuple of float
        (accuracy, -n_features)
    """
    individual_array = np.array(individual)
    n_selected = np.sum(individual_array)
    
    if n_selected == 0:
        return (0.0, 0.0)
    
    # Objective 1: Accuracy
    feature_mask = individual_array.astype(bool)
    X_selected = X_train_scaled.iloc[:, feature_mask]
    
    model = LogisticRegression(max_iter=1000, random_state=42)
    scores = cross_val_score(model, X_selected, y_train, cv=3, scoring='accuracy')
    accuracy = scores.mean()
    
    # Objective 2: Parsimony (fewer features is better)
    # Return negative to convert minimization to maximization
    parsimony = -n_selected
    
    return (accuracy, parsimony)

# Create new toolbox for multi-objective
toolbox_mo = base.Toolbox()
toolbox_mo.register("attr_bool", random.randint, 0, 1)
toolbox_mo.register(
    "individual",
    tools.initRepeat,
    creator.IndividualMulti,
    toolbox_mo.attr_bool,
    n=N_FEATURES
)
toolbox_mo.register("population", tools.initRepeat, list, toolbox_mo.individual)
toolbox_mo.register("evaluate", evaluate_multi_objective)
toolbox_mo.register("mate", tools.cxTwoPoint)
toolbox_mo.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox_mo.register("select", tools.selNSGA2)  # NSGA-II selection

# Run NSGA-II
print("Running NSGA-II for multi-objective optimization...")
print("Objectives: (1) Maximize accuracy, (2) Minimize features\n")

population_mo = toolbox_mo.population(n=50)

# Simplified NSGA-II loop (for demonstration)
NGEN_MO = 30

for gen in range(NGEN_MO):
    # Evaluate population
    fitnesses = map(toolbox_mo.evaluate, population_mo)
    for ind, fit in zip(population_mo, fitnesses):
        ind.fitness.values = fit
    
    # Select next generation
    population_mo = toolbox_mo.select(population_mo, len(population_mo))
    
    # Generate offspring
    offspring = [toolbox_mo.clone(ind) for ind in population_mo]
    
    # Apply crossover and mutation
    for i in range(1, len(offspring), 2):
        if random.random() < 0.7:
            offspring[i-1], offspring[i] = toolbox_mo.mate(offspring[i-1], offspring[i])
            del offspring[i-1].fitness.values
            del offspring[i].fitness.values
    
    for mutant in offspring:
        if random.random() < 0.2:
            toolbox_mo.mutate(mutant)
            del mutant.fitness.values
    
    # Evaluate offspring
    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = map(toolbox_mo.evaluate, invalid_ind)
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit
    
    population_mo = offspring
    
    if (gen + 1) % 10 == 0:
        print(f"Generation {gen + 1}/{NGEN_MO}")

print("\nNSGA-II completed!")
print("\nPareto Front (top 10 solutions):")
print("="*70)

# Extract Pareto front
pareto_front = tools.sortNondominated(population_mo, len(population_mo), first_front_only=True)[0]

for i, ind in enumerate(pareto_front[:10], 1):
    accuracy = ind.fitness.values[0]
    n_features = -ind.fitness.values[1]
    print(f"{i}. Accuracy: {accuracy:.4f}, Features: {int(n_features)}")

### 4.3 Visualize Pareto Front

In [ ]:
# Purpose: Visualize trade-off between objectives
# Key Concept: Pareto front shows optimal trade-offs

# Extract all population objectives
all_accuracy = [ind.fitness.values[0] for ind in population_mo]
all_n_features = [-ind.fitness.values[1] for ind in population_mo]

# Extract Pareto front objectives
pareto_accuracy = [ind.fitness.values[0] for ind in pareto_front]
pareto_n_features = [-ind.fitness.values[1] for ind in pareto_front]

# Plot
plt.figure(figsize=(12, 6))

plt.scatter(all_n_features, all_accuracy, alpha=0.5, s=50, 
           label='All solutions', color='lightblue', edgecolor='black')
plt.scatter(pareto_n_features, pareto_accuracy, alpha=0.9, s=100,
           label='Pareto front', color='red', edgecolor='black', marker='*')

plt.xlabel('Number of Features', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Multi-Objective Optimization: Pareto Front', fontsize=14)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  - Red stars show Pareto-optimal solutions")
print("  - Each star represents a different accuracy/features trade-off")
print("  - Moving right increases features, moving up increases accuracy")
print("  - Choose solution based on your priorities")

## 5. Production Pipeline with DEAP

### Key Concept: Create reusable, production-ready GA pipeline.

In [ ]:
# Purpose: Create production-ready GA feature selector
# Key Concept: Encapsulate GA in sklearn-compatible class

from sklearn.base import BaseEstimator, TransformerMixin

class DEAPFeatureSelector(BaseEstimator, TransformerMixin):
    """
    DEAP-based feature selector compatible with sklearn pipelines.
    """
    
    def __init__(self,
                 pop_size=50,
                 n_generations=40,
                 cx_prob=0.7,
                 mut_prob=0.2,
                 mut_indpb=0.05,
                 tournament_size=3,
                 random_state=42):
        self.pop_size = pop_size
        self.n_generations = n_generations
        self.cx_prob = cx_prob
        self.mut_prob = mut_prob
        self.mut_indpb = mut_indpb
        self.tournament_size = tournament_size
        self.random_state = random_state
        
        self.selected_features_ = None
        self.best_fitness_ = None
        self.logbook_ = None
    
    def fit(self, X, y):
        """
        Run GA to select features.
        
        Parameters
        ----------
        X : array-like, shape (n_samples, n_features)
            Training data
        y : array-like, shape (n_samples,)
            Target values
        
        Returns
        -------
        self : DEAPFeatureSelector
        """
        # Set random seed
        random.seed(self.random_state)
        np.random.seed(self.random_state)
        
        # Convert to DataFrame if needed
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        
        # Store data for fitness evaluation
        self.X_ = X
        self.y_ = y
        
        # Setup DEAP (simplified - use existing creator types)
        toolbox_fit = base.Toolbox()
        toolbox_fit.register("attr_bool", random.randint, 0, 1)
        toolbox_fit.register(
            "individual",
            tools.initRepeat,
            creator.Individual,
            toolbox_fit.attr_bool,
            n=X.shape[1]
        )
        toolbox_fit.register("population", tools.initRepeat, list, toolbox_fit.individual)
        toolbox_fit.register("evaluate", self._fitness)
        toolbox_fit.register("mate", tools.cxTwoPoint)
        toolbox_fit.register("mutate", tools.mutFlipBit, indpb=self.mut_indpb)
        toolbox_fit.register("select", tools.selTournament, tournsize=self.tournament_size)
        
        # Setup stats and hall of fame
        stats_fit = tools.Statistics(key=lambda ind: ind.fitness.values)
        stats_fit.register("max", np.max)
        hof_fit = tools.HallOfFame(1)
        
        # Run GA
        pop = toolbox_fit.population(n=self.pop_size)
        pop, log = algorithms.eaSimple(
            pop, toolbox_fit,
            cxpb=self.cx_prob,
            mutpb=self.mut_prob,
            ngen=self.n_generations,
            stats=stats_fit,
            halloffame=hof_fit,
            verbose=False
        )
        
        # Store results
        best_ind = hof_fit[0]
        self.selected_features_ = [i for i, gene in enumerate(best_ind) if gene == 1]
        self.best_fitness_ = best_ind.fitness.values[0]
        self.logbook_ = log
        
        return self
    
    def transform(self, X):
        """
        Select features.
        
        Parameters
        ----------
        X : array-like, shape (n_samples, n_features)
            Data to transform
        
        Returns
        -------
        X_selected : array-like, shape (n_samples, n_selected_features)
            Transformed data
        """
        if self.selected_features_ is None:
            raise ValueError("Must call fit() before transform()")
        
        if isinstance(X, pd.DataFrame):
            return X.iloc[:, self.selected_features_]
        else:
            return X[:, self.selected_features_]
    
    def _fitness(self, individual):
        """Internal fitness function."""
        individual_array = np.array(individual)
        
        if np.sum(individual_array) == 0:
            return (-999.0,)
        
        feature_mask = individual_array.astype(bool)
        X_selected = self.X_.iloc[:, feature_mask]
        
        model = LogisticRegression(max_iter=1000, random_state=42)
        scores = cross_val_score(model, X_selected, self.y_, cv=5, scoring='accuracy')
        
        accuracy = scores.mean()
        parsimony = 0.01 * (np.sum(individual_array) / len(individual_array))
        
        return (accuracy - parsimony,)

print("DEAPFeatureSelector class defined!")

### 5.1 Test Production Pipeline

In [ ]:
# Purpose: Test sklearn-compatible GA selector
# Key Concept: Integrate with sklearn pipelines

from sklearn.pipeline import Pipeline

# Create pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', DEAPFeatureSelector(
        pop_size=30,
        n_generations=20,
        random_state=42
    )),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

print("Fitting pipeline with GA feature selection...")
pipeline.fit(X_train, y_train)

# Evaluate
train_score = pipeline.score(X_train, y_train)
test_score = pipeline.score(X_test, y_test)

selector = pipeline.named_steps['selector']

print("\nPipeline Results:")
print("="*70)
print(f"Features selected: {len(selector.selected_features_)}/{X_train.shape[1]}")
print(f"Best GA fitness: {selector.best_fitness_:.4f}")
print(f"Train accuracy: {train_score:.4f}")
print(f"Test accuracy: {test_score:.4f}")
print(f"\nSelected features: {X_train.columns[selector.selected_features_].tolist()}")

## 6. Exercises

### Exercise 6.1: Custom Operator in DEAP

**Task**: Implement and register a custom crossover operator (e.g., half-uniform) in the DEAP toolbox and compare performance with built-in operators.

**Expected Output**: Comparison of convergence with custom operator.

In [ ]:
# YOUR CODE HERE
# ---------------

def custom_half_uniform_crossover(ind1, ind2):
    """
    Custom HUX crossover for DEAP.
    
    Must modify individuals in-place and return them.
    """
    # TODO: Implement
    pass

# Register in toolbox
# toolbox.register("mate", custom_half_uniform_crossover)

### Exercise 6.2: Parallel Evaluation

**Task**: Use DEAP's parallelization features to speed up fitness evaluation.

**Hint**: Use `multiprocessing.Pool` with `toolbox.register("map", pool.map)`.

In [ ]:
# YOUR CODE HERE
# ---------------

# from multiprocessing import Pool
# pool = Pool()
# toolbox.register("map", pool.map)

### Exercise 6.3: Export and Import Best Solutions

**Task**: Save the hall of fame to disk (JSON or pickle) and load it later to resume evolution or apply to new data.

In [ ]:
# YOUR CODE HERE
# ---------------

import json

# TODO: Save hall of fame
# TODO: Load and reconstruct individuals

## 7. Summary

### Key Takeaways

1. **DEAP Architecture**: Creator → Toolbox → Algorithms → Results
2. **Built-in Algorithms**: eaSimple (standard GA), eaMuPlusLambda (elitist), NSGA-II (multi-objective)
3. **Statistics & Tracking**: Logbook and HallOfFame for monitoring evolution
4. **Multi-Objective**: NSGA-II finds Pareto-optimal trade-offs
5. **Production Use**: sklearn-compatible transformers for real applications
6. **Customization**: Easy to register custom operators and fitness functions

### DEAP Best Practices

- Define Creator types once per session (global state)
- Always return tuples from fitness functions
- Use HallOfFame to preserve best solutions
- Log statistics for analysis and debugging
- Leverage built-in algorithms when possible
- Test custom operators thoroughly

### Production Checklist

- [ ] Fitness function properly validated
- [ ] Cross-validation prevents overfitting
- [ ] Hyperparameters tuned (population, generations, operators)
- [ ] Results reproducible (random seed set)
- [ ] Best solutions saved for deployment
- [ ] Performance benchmarked against baselines

### Additional Resources

- **DEAP Documentation**: https://deap.readthedocs.io/
- **DEAP Examples**: https://github.com/DEAP/deap/tree/master/examples
- **NSGA-II Paper**: Deb et al. (2002)
- **Feature Selection with GAs**: Review by Xue et al. (2016)